# MarineMamba GPU Runner

Runs all GPU experiments. Upload your `data/processed/` folder to Google Drive first.

- **Models C, D, E**: Use T4 (free tier)
- **Model F (Evo 2)**: Use A100 (Colab Pro)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kluless13/marinemamba/blob/main/notebooks/gpu_runner.ipynb)

In [1]:
# Fresh setup
!pip install mamba-ssm --no-build-isolation 2>&1 | tail -3
!pip install causal-conv1d --no-build-isolation 2>&1 | tail -3
!pip install lightning einops timm torchtext transformers huggingface_hub 2>&1 | tail -3
!git clone https://github.com/kluless13/marinemamba.git 2>/dev/null; true
%cd /content/marinemamba

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

from mamba_ssm.modules.mamba2 import Mamba2
from causal_conv1d import causal_conv1d_fn
print("mamba-ssm: OK")
print("causal-conv1d: OK")

Successfully built mamba-ssm
Successfully built causal-conv1d
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 70.8 MB/s eta 0:00:00
/content/marinemamba
GPU: NVIDIA A100-SXM4-40GB
PyTorch: 2.10.0+cu128
mamba-ssm: OK
causal-conv1d: OK


## Model C: Direct Transfer (insect → fish) — ~1 hour on T4

In [2]:
!python scripts/04_barcodemamba_models.py --mode transfer --data-dir data/processed --output-dir results

Streaming output truncated to the last 5000 lines.
                                                               train/acc: 0.000 
                                                               val/loss: 6.229  
Epoch 10/19 ━━━━━━━━━━━━━╺━━ 368/445 0:00:16 •       22.33it/s v_num: 0.000     
                                     0:00:04                   train/loss: 6.135
                                                               train/acc: 0.000 
                                                               val/loss: 6.229  
Epoch 10/19 ━━━━━━━━━━━━━╺━━ 368/445 0:00:16 •       22.33it/s v_num: 0.000     
                                     0:00:04                   train/loss: 5.743
                                                               train/acc: 0.094 
                                                               val/loss: 6.229  
Epoch 10/19 ━━━━━━━━━━━━━╺━━ 369/445 0:00:16 •       22.33it/s v_num: 0.000     
                                     0:00:04              

## Model E: Domain Adaptation (insect → fish pretrain → fine-tune) — ~8-12 hours on T4

In [3]:
!python scripts/04_barcodemamba_models.py --mode adapt --data-dir data/processed --output-dir results

Streaming output truncated to the last 5000 lines.
                                                               train/acc: 0.969 
                                                               val/loss: 1.085  
Epoch 18/19 ━━━━━━━━━━━━━╺━━ 368/445 0:00:16 •       22.34it/s v_num: 1.000     
                                     0:00:04                   train/loss: 0.542
                                                               train/acc: 0.812 
                                                               val/loss: 1.085  
Epoch 18/19 ━━━━━━━━━━━━━╺━━ 369/445 0:00:16 •       22.34it/s v_num: 1.000     
                                     0:00:04                   train/loss: 0.542
                                                               train/acc: 0.812 
                                                               val/loss: 1.085  
Epoch 18/19 ━━━━━━━━━━━━━╺━━ 369/445 0:00:16 •       22.34it/s v_num: 1.000     
                                     0:00:04              

## Model D: Scratch Pretrain — ~12-20 hours on T4 (run overnight)

In [4]:
!python scripts/04_barcodemamba_models.py --mode scratch --data-dir data/processed --output-dir results

Streaming output truncated to the last 5000 lines.
                                     0:00:04                   train/loss: 0.240
                                                               train/acc: 0.938 
                                                               val/loss: 0.756  
Epoch 12/19 ━━━━━━━━━━━━━╺━━ 369/445 0:00:16 •       22.43it/s v_num: 1.000     
                                     0:00:04                   train/loss: 0.240
                                                               train/acc: 0.938 
                                                               val/loss: 0.756  
Epoch 12/19 ━━━━━━━━━━━━━╺━━ 369/445 0:00:16 •       22.43it/s v_num: 1.000     
                                     0:00:04                   train/loss: 0.115
                                                               train/acc: 0.969 
                                                               val/loss: 0.756  
Epoch 12/19 ━━━━━━━━━━━━━╺━━ 370/445 0:00:16 •       22.44

## Model F: Evo 2 Embeddings — ~2-3 hours on A100 (requires Colab Pro)

**Switch runtime to A100 before running this cell.**

In [6]:
!pip install torchvision --upgrade --quiet
from transformers import AutoModel
model = AutoModel.from_pretrained("LongSafari/hyenadna-tiny-1k-seqlen-hf", trust_remote_code=True)
print("HyenaDNA loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/

RuntimeError: Failed to import transformers.models.timm_wrapper.configuration_timm_wrapper because of the following error (look up to see its traceback):
operator torchvision::nms does not exist

In [3]:
from helical import HyenaDNA, HyenaDNAConfig
config = HyenaDNAConfig(model_name="hyenadna-tiny-1k-seqlen")
model = HyenaDNA(configurer=config)
print("HyenaDNA: OK")

ImportError: cannot import name 'HyenaDNA' from 'helical' (/usr/local/lib/python3.12/dist-packages/helical/__init__.py)

In [1]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

# Try Evo 2 install now that PyTorch is downgraded
!pip install evo2 flash-attn --no-build-isolation 2>&1 | tail -5

# Clone repo and set up data
!git clone https://github.com/kluless13/marinemamba.git 2>/dev/null; true
%cd /content/marinemamba

# Test both models
try:
    from helical import HyenaDNA, HyenaDNAConfig
    print("HyenaDNA: OK")
except Exception as e:
    print(f"HyenaDNA: {e}")

try:
    from evo2 import Evo2
    print("Evo 2: OK")
except Exception as e:
    print(f"Evo 2: {e}")

PyTorch: 2.7.0+cu126
CUDA: 12.6
× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
/content/marinemamba
HyenaDNA: cannot import name 'HyenaDNA' from 'helical' (/usr/local/lib/python3.12/dist-packages/helical/__init__.py)
Evo 2: No module named 'evo2'


In [9]:
!pip install helical

# Quick test
from helical import HyenaDNA, HyenaDNAConfig
config = HyenaDNAConfig(model_name="hyenadna-tiny-1k-seqlen")
model = HyenaDNA(configurer=config)
print("HyenaDNA loaded!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 68.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 114.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of leidenalg to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 48.5 MB/s 

ImportError: cannot import name 'HyenaDNA' from 'helical' (/usr/local/lib/python3.12/dist-packages/helical/__init__.py)

In [5]:
!python scripts/05_evo2_embeddings.py --data-dir data/processed --output-dir results

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 72.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 70.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
MODEL F: Evo 2 Embeddings + Linear Classifier
Train: 14232 | Test: 4067 | Unseen: 1489
Traceback (most recent call last):
  File "/content/marinemamba/scripts/05_evo2_embeddings.py", line 215, in <module>
    main()
 

## Save Results to Drive

In [ ]:
!cp -r results/ /content/drive/MyDrive/marinemamba/results/
!cp -r checkpoints/ /content/drive/MyDrive/marinemamba/checkpoints/
print("Results saved to Google Drive")

# Show all results
import json, glob
for f in sorted(glob.glob("results/*.json")):
    print(f"\n{'=' * 60}")
    print(f)
    print('=' * 60)
    print(json.dumps(json.load(open(f)), indent=2))